# 14 - Recognition check for BOTH structure-loss checkpoints

**T4 GPU; Run All. No training.** After a deleted runtime restore images with notebook 02 and NBIS with notebook 04, on the SAME runtime, before running this notebook. Do not retrain 07-13. Keep completed reports from 08/11 and training artifacts from 13 on Drive. Run only one copy at a time.

Notebook 13's selected Sobel model had worse overall patch L1 and original-image preservation than its L1 control. Do not discard this negative result or claim failure/success of recognition from MAE. Compare input, M0, l1_control and l1_sobel on ALL locked validation probes, all ten conditions and unchanged original galleries. No checkpoint re-selection or coefficient tuning.

Use the hash-verified NBIS/inference/metric helpers archived by notebook 11 (originally exported by 08). Full native normalized-size FP32 inference, no AMP/TF32/tiles, round to 8-bit PNG at 500 dpi. The losses are not needed for inference. Reuse 11's recorded input/M0 units only if matcher binaries, libraries, GPU, precision and pair/simulator provenance match; otherwise explicitly recompute both references. New original-gallery templates must agree before old scores can be reused.

Normally 20 NEW variants are evaluated and 20 references reused; incompatible references trigger all 40. NBIS extraction/matching uses CPU. Each completed variant is persisted with checksums, portable PNG/template/log archives and a completion marker. Run All resumes completed variants; an unfinished variant may need to repeat extraction after runtime deletion.

Primary: original-input thresholds fixed per database across all methods and conditions. Secondary: each condition/method's validation-FMR<=1% calibration, NOT necessarily equal achieved FMR/FAR and not deployable blind calibration. Report FAR together with FRR, conditional FMR/FNMR, low-minutiae rejections, approximate EER, byte-distinct sensitivity, original-image harms, and the direct l1_sobel minus l1_control comparison. Reference and enhanced probes always use unchanged original galleries.

This validation has been repeatedly used for model selection and development. No independent-test improvement, real dry/wet labels, statistical significance, novelty or final generalization claims are made. FVC2004 and test_synthetic remain sealed. Do not add a GAN or continue tuning Sobel lambda on these same validation results if the controlled objective change is unhelpful.

Outputs: MyDrive/FingerprintResearch/evaluations/m2_structure_validation_v1/<configuration-hash>/. Completion marker: STRUCTURE VALIDATION MATCHING COMPLETE. Send both fixed/calibrated summaries and original-image harms before deciding the next research direction.

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import inspect
import json
import math
import re
import shutil
import subprocess
import sys
import types
import zipfile
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from google.colab import drive
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU kernel. A T4 is sufficient.')
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
M0_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'
M2_DIR = PROJECT_ROOT / 'experiments' / 'm2_structure_loss_v1_seed20261110' / '92579202157d85254b7f'
REFERENCE_DIR = PROJECT_ROOT / 'evaluations' / 'm1_controlled_validation_v1' / 'bef0c1363b1e6a721afc'
NBIS_ROOT = LOCAL_ROOT / 'tools' / 'nbis_5_0_0'
MINDTCT, BOZORTH3 = (NBIS_ROOT / 'install' / 'bin' / name for name in ('mindtct', 'bozorth3'))
METHODS = ('input', 'm0', 'l1_control', 'l1_sobel')
CONDITIONS = [('identity', 'none')] + [(family, severity)
    for family in ('dry_like', 'wet_like', 'generic') for severity in ('mild', 'moderate', 'severe')]
COMPARISONS = [('input', 'm0'), ('input', 'l1_control'), ('input', 'l1_sobel'), ('l1_control', 'l1_sobel')]

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))

def verify_files(directory, hashes):
    for name, expected in hashes.items():
        if file_digest(directory / name) != expected:
            raise RuntimeError(f'Frozen artifact checksum mismatch: {directory / name}')

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot load module: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module
print('GPU:', torch.cuda.get_device_name(0), '| Evaluate BOTH fixed checkpoints; no training.')

In [ ]:
split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))
verify_files(SPLIT_DIR, split_protocol['file_sha256'])
locked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)
validation = locked[locked['split'].eq('validation')].sort_values('relative_path').reset_index(drop=True)
if validation.empty or set(validation['dataset']) != {'FVC2000', 'FVC2002'} or validation.groupby(['dataset', 'db']).ngroups != 6:
    raise RuntimeError('Expected complete source-only validation in six databases.')
pair_config = json.loads((PAIR_DIR / 'configuration.json').read_text(encoding='utf-8'))
sim_config = json.loads((SIM_DIR / 'configuration.json').read_text(encoding='utf-8'))
m0_config = json.loads((M0_DIR / 'configuration.json').read_text(encoding='utf-8'))
m2_config = json.loads((M2_DIR / 'configuration.json').read_text(encoding='utf-8'))
m2_completion = json.loads((M2_DIR / 'completion.json').read_text(encoding='utf-8'))
reference_config = json.loads((REFERENCE_DIR / 'configuration.json').read_text(encoding='utf-8'))
reference_completion = json.loads((REFERENCE_DIR / 'completion.json').read_text(encoding='utf-8'))
m0_best = json.loads((M0_DIR / 'best.json').read_text(encoding='utf-8'))
m0_completion = json.loads((M0_DIR / 'completion.json').read_text(encoding='utf-8'))
m0_sha, m2_sha, reference_sha = (file_digest(directory / 'configuration.json') for directory in (M0_DIR, M2_DIR, REFERENCE_DIR))
verify_files(M0_DIR, m0_config['source_file_sha256'])
verify_files(M2_DIR, m2_config['source_file_sha256'])
verify_files(M2_DIR, m2_completion['file_sha256'])
verify_files(REFERENCE_DIR, reference_completion['file_sha256'])
if any(config['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json') for config in (pair_config, sim_config, m0_config)):
    raise RuntimeError('Split provenance differs across notebooks.')
if (file_digest(PAIR_DIR / 'pairs.csv') != pair_config['pair_csv_sha256']
        or sim_config['normalization_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or m0_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or m2_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or file_digest(SIM_DIR / 'fingerprint_degradation.py') != sim_config['module_sha256']
        or m0_config['simulator_module_sha256'] != sim_config['module_sha256']):
    raise RuntimeError('Frozen pairs/simulator provenance mismatch.')
if (m0_completion['status'] != 'm0_training_complete' or m0_completion['configuration_sha256'] != m0_sha
        or m0_completion['epochs_completed'] != m0_config['epochs'] or m0_completion['best_checkpoint'] != m0_best
        or m0_best['configuration_sha256'] != m0_sha or file_digest(M0_DIR / m0_best['file']) != m0_best['sha256']):
    raise RuntimeError('M0 completion/checkpoint provenance mismatch.')
if (m2_completion['status'] != 'structure_loss_control_complete' or m2_completion['configuration_sha256'] != m2_sha
        or m2_config['m0_configuration_sha256'] != m0_sha or set(m2_config['run_names']) != {'l1_control', 'l1_sobel'}):
    raise RuntimeError('M2 completed loss experiment provenance mismatch.')
for name in ('restoration_baseline.py', 'train_sources.csv', 'validation_sources.csv'):
    if file_digest(M2_DIR / name) != file_digest(M0_DIR / name):
        raise RuntimeError(f'M2 model/source differs from M0: {name}')
if (reference_completion['status'] != 'm1_controlled_validation_matching_complete'
        or reference_completion['configuration_sha256'] != reference_sha
        or reference_config['m0_configuration_sha256'] != m0_sha
        or reference_config['model_checkpoints']['m0'] != m0_best
        or reference_config['pair_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or reference_config['pair_csv_sha256'] != pair_config['pair_csv_sha256']
        or reference_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')):
    raise RuntimeError('Notebook-11 reference provenance mismatch.')
helper_path = REFERENCE_DIR / 'frozen_nb08_helpers.py.txt'
if file_digest(helper_path) != reference_config['frozen_helpers_sha256']:
    raise RuntimeError('Frozen NBIS/inference/metric helper source changed.')
HELPER_SOURCE = helper_path.read_bytes().decode('utf-8')
helper_code = compile(HELPER_SOURCE, str(helper_path), 'exec')
for config in (m0_config, m2_config):
    if (str(torch.__version__) != config['torch_version'] or str(torch.version.cuda) != config['torch_cuda']
            or Image.__version__ != config['pillow_version'] or np.__version__ != config['numpy_version']):
        raise RuntimeError('Runtime library versions differ from frozen training.')
if Image.__version__ != pair_config['pillow_version'] or Image.__version__ != sim_config['pillow_version'] or np.__version__ != sim_config['numpy_version']:
    raise RuntimeError('Normalization/simulator library versions differ.')
if not (NBIS_ROOT / 'verified_build.json').is_file():
    raise FileNotFoundError('NBIS runtime installation missing. Run notebook 04 first on this runtime.')
build = json.loads((NBIS_ROOT / 'verified_build.json').read_text(encoding='utf-8'))
if (build['signature']['source_sha256'] != pair_config['nbis_source_sha256']
        or pair_config['nbis_source_sha256'] != '0adf8ab0f6b0e4208de50ca00ba21d3d77112ecd66288757ddfed21f6bee92c3'
        or pair_config['mindtct_flags'] != [] or pair_config['bozorth3_flags'] != ['-A', 'minminutiae=10']):
    raise RuntimeError('NBIS source/settings differ from the frozen pair protocol.')
for binary in (MINDTCT, BOZORTH3):
    if not binary.is_file() or file_digest(binary) != build['binary_sha256'][binary.name]:
        raise RuntimeError(f'NBIS binary missing/modified: {binary}')
    subprocess.run([str(binary), '-version'], capture_output=True, text=True, check=True)
pairs = pd.read_csv(PAIR_DIR / 'pairs.csv', keep_default_na=False)
if pairs['pair_id'].duplicated().any() or validation['relative_path'].duplicated().any() or pairs['duplicate_sensitivity_eligible'].dtype != bool:
    raise RuntimeError('Invalid pair/image keys or duplicate-sensitivity encoding.')
probe_records = validation[validation['impression_id'].ne(1)].to_dict('records')
gallery_records = validation[validation['impression_id'].eq(1)].to_dict('records')
if (set(pairs['probe_path']) != {row['relative_path'] for row in probe_records}
        or set(pairs['gallery_path']) != {row['relative_path'] for row in gallery_records}):
    raise RuntimeError('Frozen pairs do not cover all validation probes/galleries.')
for record in validation.to_dict('records'):
    path = DATA_ROOT / record['relative_path']
    if not path.is_file() or file_digest(path) != record['sha256']:
        raise RuntimeError(f'Image missing/changed: {path}. Restore with notebook 02 on this runtime.')
training = load_module('m2_matching_model_v1', M2_DIR / 'restoration_baseline.py')
simulator = load_module('m2_matching_simulator_v1', SIM_DIR / 'fingerprint_degradation.py')
models, model_pointers = {}, {'m0': m0_best}
for name in ('l1_control', 'l1_sobel'):
    directory = M2_DIR / name
    if file_digest(directory / 'completion.json') != m2_completion['run_completion_sha256'][name]:
        raise RuntimeError(f'M2 run completion changed: {name}')
    marker = json.loads((directory / 'completion.json').read_text(encoding='utf-8'))
    verify_files(directory, marker['file_sha256'])
    pointer = json.loads((directory / 'best.json').read_text(encoding='utf-8'))
    if (marker['configuration_sha256'] != m2_sha or marker['run'] != name
            or marker['epochs_completed'] != len(m2_config['epoch_conditions'])
            or marker['result']['best_checkpoint'] != pointer or pointer['configuration_sha256'] != m2_sha
            or file_digest(directory / pointer['file']) != pointer['sha256']):
        raise RuntimeError(f'M2 best checkpoint/completion mismatch: {name}')
    model_pointers[name] = pointer
for name in ('m0', 'l1_control', 'l1_sobel'):
    directory = M0_DIR if name == 'm0' else M2_DIR / name
    state = torch.load(directory / model_pointers[name]['file'], map_location='cpu', weights_only=True)
    expected_sha = m0_sha if name == 'm0' else m2_sha
    if state['configuration_sha256'] != expected_sha:
        raise RuntimeError(f'Model payload configuration mismatch: {name}')
    if (name == 'm0' and state['epoch'] != m0_best['epoch']) or (name != 'm0' and
        (state['run'] != name or state['completed_epochs'] != model_pointers[name]['completed_epochs'])):
        raise RuntimeError(f'Model payload checkpoint selection mismatch: {name}')
    model = training.RestorationUNet(base=m0_config['base_channels']).to(DEVICE)
    model.load_state_dict(state['model'], strict=True)
    model.eval()
    models[name] = model
del state
current_reference_settings = {'nbis_binary_sha256': build['binary_sha256'], 'torch_version': str(torch.__version__),
    'torch_cuda': str(torch.version.cuda), 'numpy_version': np.__version__, 'pillow_version': Image.__version__,
    'gpu_name': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None,
    'gpu_compute_capability': list(torch.cuda.get_device_capability(0)) if DEVICE.type == 'cuda' else None,
    'inference': 'one full native normalized image, FP32, no AMP, no padding, no tiles',
    'quantization': 'numpy.rint(image * 255) to uint8; lossless PNG at 500dpi', 'tf32_enabled': False}
reference_mismatches = [name for name, value in current_reference_settings.items()
                        if reference_config['inference_and_matcher'].get(name) != value]
REUSE_REFERENCES = not reference_mismatches
print('Locked probes:', len(probe_records), '| Pairs per variant:', len(pairs))
print('Reuse recorded notebook-11 input/M0 references:', REUSE_REFERENCES)
if reference_mismatches:
    print('Reference compatibility differs:', reference_mismatches, '; input AND M0 will be explicitly recomputed.')

In [ ]:
def make_helpers(configuration_sha):
    module = types.ModuleType('frozen_structure_matching_helpers')
    names = ('torch', 'np', 'pd', 'Image', 'Path', 'hashlib', 'json', 'math', 're', 'shutil', 'subprocess', 'zipfile',
        'file_digest', 'json_bytes', 'write_verified', 'training', 'simulator', 'DATA_ROOT', 'pair_config',
        'sim_config', 'pairs', 'probe_records', 'MINDTCT', 'BOZORTH3', 'DEVICE')
    module.__dict__.update({name: globals()[name] for name in names})
    module.evaluation_sha, module.TEMPLATE_DIR, module.model = configuration_sha, TEMPLATE_DIR, None
    exec(helper_code, module.__dict__)
    return module

def get_unit(family, severity, method):
    if method not in METHODS:
        raise ValueError(f'Unknown method: {method}')
    unit = f'{family}_{severity}_{method}'
    if unit in UNIT_CACHE:
        return UNIT_CACHE[unit]
    reused = REUSE_REFERENCES and method in ('input', 'm0')
    if reused:
        provenance = reference_completion['unit_provenance'][unit]
        directory = Path(provenance['directory'])
        if file_digest(directory / 'completion.json') != provenance['completion_sha256']:
            raise RuntimeError(f'Frozen reference unit marker changed: {unit}')
        expected_sha = reference_config['reference_evaluation_sha256'] if provenance['origin'] == 'notebook08' else reference_sha
        if expected_sha not in reference_helpers:
            reference_helpers[expected_sha] = make_helpers(expected_sha)
        value = reference_helpers[expected_sha].load_unit(directory)
        if value is None:
            raise RuntimeError(f'Completed reference unit missing: {unit}')
    else:
        directory = REPORT_DIR / unit
        value = active_helpers.load_unit(directory)
    if value is not None:
        UNIT_CACHE[unit] = value
        UNIT_PROVENANCE[unit] = {'condition': f'{family}/{severity}', 'method': method,
            'origin': 'prior_verified_reference' if reused else 'notebook14', 'directory': str(directory),
            'completion_sha256': file_digest(directory / 'completion.json')}
    return value

def paired_rows(before, after, thresholds, condition, baseline_method, method):
    rows = active_helpers.comparison_rows(before, after, thresholds, condition)
    for row in rows:
        row['baseline_method'], row['method'] = baseline_method, method
        row['baseline_attempt_frr'] = row.pop('input_attempt_frr')
        row['method_attempt_frr'] = row.pop('m0_attempt_frr')
    return rows

In [ ]:
adapter_source = '\n\n'.join(inspect.getsource(function) for function in (make_helpers, get_unit, paired_rows))
evaluation_config = {'version': 'm2_structure_validation_v1', 'm0_configuration_sha256': m0_sha,
    'm2_configuration_sha256': m2_sha, 'model_checkpoints': model_pointers,
    'reference_evaluation_sha256': reference_sha, 'reference_completion_sha256': file_digest(REFERENCE_DIR / 'completion.json'),
    'frozen_helpers_sha256': file_digest(helper_path), 'adapters_sha256': hashlib.sha256(adapter_source.encode()).hexdigest(),
    'pair_configuration_sha256': file_digest(PAIR_DIR / 'configuration.json'), 'pair_csv_sha256': pair_config['pair_csv_sha256'],
    'simulator_configuration_sha256': file_digest(SIM_DIR / 'configuration.json'),
    'methods': METHODS, 'conditions': CONDITIONS, 'comparisons': COMPARISONS,
    'reference_reuse': REUSE_REFERENCES, 'reference_mismatches': reference_mismatches,
    'inference_and_matcher': current_reference_settings,
    'thresholds': 'original input fixed per db primary; condition/method validation FMR<=.01 secondary',
    'scope': 'all locked validation pairs; original galleries; no test evaluation'}
evaluation_data = json_bytes(evaluation_config)
evaluation_sha = hashlib.sha256(evaluation_data).hexdigest()
REPORT_DIR = PROJECT_ROOT / 'evaluations' / 'm2_structure_validation_v1' / evaluation_sha[:20]
CACHE_DIR = LOCAL_ROOT / 'm2_structure_validation_v1' / evaluation_sha[:20]
TEMPLATE_DIR = CACHE_DIR / 'templates'
for directory in (REPORT_DIR, CACHE_DIR, TEMPLATE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
frozen_files = {'configuration.json': evaluation_data, 'frozen_matching_helpers.py.txt': HELPER_SOURCE.encode(),
                'evaluation_adapters.py.txt': adapter_source.encode()}
for name, data in frozen_files.items():
    path = REPORT_DIR / name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Frozen matching evaluation changed: {path}')
    if not path.exists():
        write_verified(path, data)
active_helpers = make_helpers(evaluation_sha)
reference_helpers, UNIT_CACHE, UNIT_PROVENANCE = {}, {}, {}
print('Frozen evaluation:', evaluation_sha)
print('Reports/resume directory:', REPORT_DIR)

In [ ]:
pending = [(family, severity, method) for family, severity in CONDITIONS for method in METHODS
           if get_unit(family, severity, method) is None]
galleries = None
if pending:
    gallery_rows = []
    for record in gallery_records:
        reference, _ = active_helpers.condition_input(record, 'identity', 'none')
        gallery_rows.append({**record, **active_helpers.extract_png(active_helpers.quantize(reference))})
    galleries = pd.DataFrame(gallery_rows)
    write_csv(REPORT_DIR / 'gallery_extraction.csv', galleries)
    if REUSE_REFERENCES:
        columns = ['relative_path', 'png_sha256', 'template_sha256', 'minutiae', 'status']
        previous = pd.read_csv(REFERENCE_DIR / 'gallery_extraction.csv')[columns].sort_values('relative_path').reset_index(drop=True)
        current = galleries[columns].sort_values('relative_path').reset_index(drop=True)
        if not current.equals(previous):
            raise RuntimeError('Original gallery extraction differs; do not mix reference scores.')
    print('Fixed original galleries prepared:', len(galleries), '| Pending variants:', len(pending), flush=True)
for family, severity in CONDITIONS:
    for method in METHODS:
        unit = f'{family}_{severity}_{method}'
        if get_unit(family, severity, method) is not None:
            print('Reused/resumed verified variant:', unit, flush=True)
            continue
        directory, work = REPORT_DIR / unit, CACHE_DIR / unit
        directory.mkdir(parents=True, exist_ok=True)
        work.mkdir(parents=True, exist_ok=True)
        if method != 'input':
            active_helpers.model = models[method]
        rows = []
        for index, record in enumerate(probe_records, start=1):
            reference, degraded = active_helpers.condition_input(record, family, severity)
            output = degraded if method == 'input' else active_helpers.restore_image(degraded)
            rows.append({**record, **active_helpers.extract_png(active_helpers.quantize(output)),
                **active_helpers.pixel_metrics(reference, degraded, output), 'condition': f'{family}/{severity}', 'method': method})
            if index % 100 == 0 or index == len(probe_records):
                write_csv(directory / 'extraction_progress.csv', pd.DataFrame(rows))
                print(f'{unit}: extracted/cached {index}/{len(probe_records)} probes', flush=True)
        extraction = pd.DataFrame(rows)
        scored = active_helpers.match_pairs(extraction, galleries, work)
        write_csv(directory / 'scores.csv', scored)
        write_csv(directory / 'extraction.csv', extraction)
        active_helpers.archive_templates(directory / 'templates_and_logs.zip', rows + galleries.to_dict('records'), work)
        marker = {'configuration_sha256': evaluation_sha, 'variant': unit,
            'file_sha256': {name: file_digest(directory / name) for name in ('scores.csv', 'extraction.csv', 'templates_and_logs.zip')}}
        write_verified(directory / 'completion.json', json_bytes(marker))
        get_unit(family, severity, method)
        print('Saved variant:', unit, '| rejected comparisons:', int((~scored['matchable']).sum()), flush=True)
write_csv(REPORT_DIR / 'unit_provenance.csv', pd.DataFrame(UNIT_PROVENANCE.values()))
print('All 40 method/condition variants verified; fixed checkpoints and original galleries unchanged.')

In [ ]:
original, _ = get_unit('identity', 'none', 'input')
fixed_thresholds = {key: active_helpers.calibrate_threshold(group.loc[group['matchable'] & group['label'].eq(0), 'score'])
                    for key, group in original.groupby(['dataset', 'db'], sort=True)}
metric_rows, pixel_rows, paired_changes, threshold_rows, curves = [], [], [], [], []
for family, severity in CONDITIONS:
    condition = f'{family}/{severity}'
    scores = {}
    for method in METHODS:
        scored, extraction = get_unit(family, severity, method)
        scores[method] = scored
        for (dataset, db), images in extraction.groupby(['dataset', 'db'], sort=True):
            pixel_rows.append({'condition': condition, 'method': method, 'dataset': dataset, 'db': db,
                'probe_images': len(images), 'insufficient_minutiae': int(images['status'].ne('ok').sum()),
                'image_failure_rate': float(images['status'].ne('ok').mean()),
                **{name: float(images[name].mean()) for name in ('float_input_mae', 'float_output_mae', 'float_change_mae',
                    'png_input_mae', 'png_output_mae', 'changed_pixel_fraction')},
                'png_identical_image_fraction': float(images['png_identical_to_input'].mean())})
        for (dataset, db), group in scored.groupby(['dataset', 'db'], sort=True):
            valid_impostors = group.loc[group['matchable'] & group['label'].eq(0), 'score']
            calibrated = active_helpers.calibrate_threshold(valid_impostors) if len(valid_impostors) else None
            available = group.loc[group['matchable'], 'label'].nunique() == 2
            eer, gap = None, None
            if available:
                curve = active_helpers.conditional_curve(group)
                point = curve.loc[(curve['conditional_fmr'] - curve['conditional_fnmr']).abs().idxmin()]
                eer = float((point['conditional_fmr'] + point['conditional_fnmr']) / 2)
                gap = float(abs(point['conditional_fmr'] - point['conditional_fnmr']))
                curves.append(curve.assign(condition=condition, method=method, dataset=dataset, db=db))
            if calibrated is None or not available:
                print(f'WARNING: conditional calibration/EER unavailable for {condition}/{method}/{dataset}/{db}; all attempts retained.')
            threshold_rows.append({'condition': condition, 'method': method, 'dataset': dataset, 'db': db,
                'original_fixed_threshold': fixed_thresholds[(dataset, db)], 'condition_calibrated_threshold': calibrated,
                'calibration_status': 'ok' if calibrated is not None else 'no_matchable_impostors'})
            for policy, threshold in (('original_fixed', fixed_thresholds[(dataset, db)]), ('condition_calibrated', calibrated)):
                if threshold is None:
                    continue
                for view, selected in (('all_impressions', group),
                                       ('byte_distinct_sensitivity', group[group['duplicate_sensitivity_eligible']])):
                    values = active_helpers.metrics_at_threshold(selected, threshold)
                    if policy == 'condition_calibrated' and view == 'all_impressions' and values['conditional_fmr'] > 0.01 + 1e-12:
                        raise RuntimeError('Condition-calibrated validation FMR exceeded 1%.')
                    metric_rows.append({'condition': condition, 'method': method, 'dataset': dataset, 'db': db,
                        'threshold_policy': policy, 'view': view, **values,
                        'conditional_eer_discrete_approx': eer if view == 'all_impressions' else None,
                        'eer_crossing_gap': gap if view == 'all_impressions' else None})
    for baseline_method, method in COMPARISONS:
        paired_changes.extend(paired_rows(scores[baseline_method], scores[method], fixed_thresholds, condition, baseline_method, method))
metrics, pixels, paired = pd.DataFrame(metric_rows), pd.DataFrame(pixel_rows), pd.DataFrame(paired_changes)
write_csv(REPORT_DIR / 'validation_metrics.csv', metrics)
write_csv(REPORT_DIR / 'pixel_and_failure_metrics.csv', pixels)
write_csv(REPORT_DIR / 'paired_genuine_changes.csv', paired)
write_csv(REPORT_DIR / 'validation_thresholds.csv', pd.DataFrame(threshold_rows))
write_csv(REPORT_DIR / 'conditional_curves.csv', pd.concat(curves, ignore_index=True))
macro = metrics[metrics['view'].eq('all_impressions')].groupby(['condition', 'method', 'threshold_policy'], sort=True).agg(
    databases=('db', 'size'), macro_attempt_frr=('attempt_frr', 'mean'), macro_attempt_far=('attempt_far', 'mean'),
    macro_conditional_fnmr=('conditional_fnmr', 'mean'), valid_fnmr_databases=('conditional_fnmr', 'count'),
    macro_conditional_fmr=('conditional_fmr', 'mean'), valid_fmr_databases=('conditional_fmr', 'count')).reset_index()
if not macro['databases'].eq(6).all():
    raise RuntimeError('Macro summaries must include all six databases.')
write_csv(REPORT_DIR / 'macro_operating_points.csv', macro)
write_csv(REPORT_DIR / 'macro_fixed_threshold_metrics.csv', macro[macro['threshold_policy'].eq('original_fixed')])
aggregate_rows = []
scopes = {'synthetic_nine': [f'{family}/{severity}' for family, severity in CONDITIONS if family != 'identity'],
    **{family: [f'{name}/{severity}' for name, severity in CONDITIONS if name == family] for family in ('dry_like', 'wet_like', 'generic')}}
for scope, members in scopes.items():
    frame = macro[macro['condition'].isin(members)]
    for (method, policy), group in frame.groupby(['method', 'threshold_policy'], sort=True):
        if len(group) != len(members):
            raise RuntimeError('Incomplete aggregate condition coverage.')
        aggregate_rows.append({'scope': scope, 'method': method, 'threshold_policy': policy,
            'conditions': len(group), **{name: float(group[name].mean()) for name in
                ('macro_attempt_frr', 'macro_attempt_far', 'macro_conditional_fnmr', 'macro_conditional_fmr')},
            'complete_conditional_metrics': bool(group['valid_fnmr_databases'].eq(6).all() and group['valid_fmr_databases'].eq(6).all())})
aggregates = pd.DataFrame(aggregate_rows)
write_csv(REPORT_DIR / 'aggregate_operating_points.csv', aggregates)
print('Equal condition/database weights; condition-calibrated FMR <= 1% is NOT identical achieved FMR or independent test performance.')
print(aggregates.to_string(index=False))
print('\nOriginal-image harm/recovery at original fixed thresholds:')
print(paired[paired['condition'].eq('identity/none')].groupby(['baseline_method', 'method'])[['acceptance_recoveries', 'acceptance_harms']].sum().to_string())
print('\nDirect Sobel versus matched L1 at original fixed thresholds:')
print(paired[paired['baseline_method'].eq('l1_control')].to_string(index=False))

In [ ]:
record = probe_records[0]
figure, axes = plt.subplots(10, 5, figsize=(15, 24))
for row, (family, severity) in enumerate(CONDITIONS):
    reference, degraded = active_helpers.condition_input(record, family, severity)
    images = [('input', degraded)]
    for method in ('m0', 'l1_control', 'l1_sobel'):
        active_helpers.model = models[method]
        images.append((method, active_helpers.restore_image(degraded)))
    images.append(('reference', reference))
    for column, (title, image) in enumerate(images):
        axes[row, column].imshow(active_helpers.quantize(image), cmap='gray', vmin=0, vmax=255)
        axes[row, column].set_title(f'{family}/{severity} | {title}', fontsize=7)
        axes[row, column].axis('off')
figure.tight_layout()
figure.savefig(REPORT_DIR / 'full_image_preview.png', dpi=140)
plt.show()
plt.close(figure)
if len(UNIT_PROVENANCE) != len(CONDITIONS) * len(METHODS):
    raise RuntimeError('Missing method/condition variants at completion.')
files = [*frozen_files, 'unit_provenance.csv', 'gallery_extraction.csv', 'validation_metrics.csv',
    'pixel_and_failure_metrics.csv', 'paired_genuine_changes.csv', 'validation_thresholds.csv', 'conditional_curves.csv',
    'macro_operating_points.csv', 'macro_fixed_threshold_metrics.csv', 'aggregate_operating_points.csv', 'full_image_preview.png']
completion = {'status': 'structure_validation_matching_complete', 'configuration_sha256': evaluation_sha,
    'file_sha256': {name: file_digest(REPORT_DIR / name) for name in files}, 'unit_provenance': UNIT_PROVENANCE,
    'methods': METHODS, 'variants_available': len(UNIT_PROVENANCE), 'pairs_per_variant': len(pairs),
    'reference_reuse': REUSE_REFERENCES, 'independent_test_improvement_verified': False, 'external_test_evaluated': False,
    'interpretation': 'exploratory validation; selected checkpoints, standard loss ablation, no significance or real moisture claims'}
write_verified(REPORT_DIR / 'completion.json', json_bytes(completion))
print('STRUCTURE VALIDATION MATCHING COMPLETE:', REPORT_DIR)
print('Send aggregate_operating_points, paired_genuine_changes and previews. Keep negative outcomes; do not tune lambda on test data.')